# LeetCode 2980 — Check if Bitwise OR Has Trailing Zeros bằng Python

Link đề: https://leetcode.com/problems/check-if-bitwise-or-has-trailing-zeros/ · Độ khó: **Easy** · Chủ đề: Array, Bit Manipulation

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def hasTrailingZeros(self, nums: List[int]) -> bool:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả vét cạn mọi tập con), nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn **hai hoặc nhiều** phần tử của `nums` sao cho **OR bit** của chúng có **ít nhất một chữ số `0` ở cuối** (trong **biểu diễn nhị phân**), và hỏi có chọn được không.
2. Hiểu **"trailing zero" nhị phân** nghĩa là bit thấp nhất bằng `0`, tức số đó **chẵn**; biết kiểm tra bằng `x & 1` hoặc `x % 2`.
3. Hiểu **quy tắc của phép OR**: bit của kết quả bằng `1` nếu **có ít nhất một** số có bit `1`; từ đó suy ra OR của một nhóm là chẵn **khi và chỉ khi mọi số trong nhóm đều chẵn**.
4. Đi từ cách **thử mọi cặp** → **đếm số chẵn, cần `>= 2`** → **dừng sớm bằng cờ** → **tổng quát hóa: nhiều bit `0` ở cuối ⇔ chia hết cho `2^t`**.
5. Tránh các lỗi kinh điển: chỉ cần **một** số chẵn (sai: phải chọn ≥ 2 phần tử), OR **tất cả** phần tử thay vì chọn nhóm, dùng `> 2` thay vì `>= 2`, và nhầm "chữ số `0` ở cuối" của hệ **thập phân** với hệ nhị phân.

## 2. Đề bài & ví dụ

Cho mảng số nguyên **dương** `nums`. Bạn phải kiểm tra xem có thể **chọn hai hoặc nhiều phần tử** từ `nums` sao cho **OR bit** (bitwise OR) của các phần tử được chọn có **ít nhất một bit `0` ở cuối** trong biểu diễn nhị phân của nó hay không.

Ví dụ, số `5` có biểu diễn nhị phân `101` **không** có bit `0` ở cuối; số `4` có biểu diễn nhị phân `100` có **hai** bit `0` ở cuối.

Trả về `True` nếu có thể chọn được như vậy, ngược lại trả về `False`.

**Ví dụ 1**

- Input: `nums = [1, 2, 3, 4, 5]`
- Output: `True`
- Giải thích: chọn `2` và `4`: `2 | 4 = 6`, nhị phân `110`, có một bit `0` ở cuối.

**Ví dụ 2**

- Input: `nums = [2, 4, 8, 16]`
- Output: `True`
- Giải thích: chọn `2` và `4`: `2 | 4 = 6 = 110`, có một bit `0` ở cuối. Các cặp khác cũng thỏa (ví dụ `8 | 16 = 24 = 11000` có ba bit `0` ở cuối).

**Ví dụ 3**

- Input: `nums = [1, 3, 5, 7, 9]`
- Output: `False`
- Giải thích: không có cách chọn nào cho OR có bit `0` ở cuối (mọi số đều lẻ).

**Ràng buộc:** `2 <= nums.length <= 100`; `1 <= nums[i] <= 100`.

In [1]:
from typing import List
from itertools import combinations
from functools import reduce
from operator import or_
import random

examples = [
    ([1, 2, 3, 4, 5], True),
    ([2, 4, 8, 16], True),
    ([1, 3, 5, 7, 9], False),
    ([1, 2], False),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [1, 2, 3, 4, 5] -> kỳ vọng True
nums = [2, 4, 8, 16] -> kỳ vọng True
nums = [1, 3, 5, 7, 9] -> kỳ vọng False
nums = [1, 2] -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Biểu diễn nhị phân và bit thấp nhất

Mỗi số nguyên dương viết được dưới dạng nhị phân. **Bit thấp nhất** (bên phải cùng) cho biết số **chẵn hay lẻ**: bit thấp nhất `0` ⇔ số chẵn; `1` ⇔ số lẻ. Hàm `bin(x)` in biểu diễn nhị phân.

In [2]:
for x in [1, 2, 3, 4, 5, 6, 8, 12]:
    print(f"{x:>2} = {bin(x):>7}  bit thấp nhất = {x & 1}  ({'lẻ' if x & 1 else 'chẵn'})")

 1 =     0b1  bit thấp nhất = 1  (lẻ)
 2 =    0b10  bit thấp nhất = 0  (chẵn)
 3 =    0b11  bit thấp nhất = 1  (lẻ)
 4 =   0b100  bit thấp nhất = 0  (chẵn)
 5 =   0b101  bit thấp nhất = 1  (lẻ)
 6 =   0b110  bit thấp nhất = 0  (chẵn)
 8 =  0b1000  bit thấp nhất = 0  (chẵn)
12 =  0b1100  bit thấp nhất = 0  (chẵn)


### 3.2. "Bit `0` ở cuối" (trailing zero)

Số có **ít nhất một bit `0` ở cuối** ⇔ bit thấp nhất của nó là `0` ⇔ nó **chẵn**. Đếm được số bit `0` ở cuối bằng `(x & -x).bit_length() - 1`: `x & -x` giữ lại bit `1` thấp nhất, và vị trí của nó chính là số bit `0` đứng phía sau.

Chú ý: "chữ số `0` ở cuối" của bài là về **nhị phân**, không phải thập phân (`10`, `20`, ...).

In [3]:
def trailing_zeros(x):
    return (x & -x).bit_length() - 1

for x in [1, 2, 6, 8, 12, 24, 5]:
    print(f"{x:>2} = {bin(x):>7} -> {trailing_zeros(x)} bit 0 ở cuối")

 1 =     0b1 -> 0 bit 0 ở cuối
 2 =    0b10 -> 1 bit 0 ở cuối
 6 =   0b110 -> 1 bit 0 ở cuối
 8 =  0b1000 -> 3 bit 0 ở cuối
12 =  0b1100 -> 2 bit 0 ở cuối
24 = 0b11000 -> 3 bit 0 ở cuối
 5 =   0b101 -> 0 bit 0 ở cuối


### 3.3. Toán tử OR `|`

`a | b` thực hiện OR **từng bit**: bit kết quả là `1` nếu **ít nhất một** trong hai bit tương ứng là `1`, và `0` chỉ khi **cả hai** đều `0`. Với nhiều số, `a | b | c` cũng vậy: một bit là `0` chỉ khi **mọi** số có bit đó bằng `0`.

In [4]:
print(bin(2), bin(4), bin(2 | 4))          # 0b10 0b100 0b110
print(bin(1), bin(2), bin(1 | 2))          # 0b1 0b10 0b11
print(bin(3 | 5 | 7))                       # 0b111

0b10 0b100 0b110
0b1 0b10 0b11
0b111


### 3.4. OR chỉ có thể "bật thêm" bit

Kết quả của OR luôn có **mọi bit `1`** mà từng số có (nó chỉ có thể **thêm** bit `1`, không thể tắt). Hệ quả cho bit thấp nhất: **OR là chẵn ⇔ mọi số trong nhóm đều chẵn**. Chỉ cần **một** số lẻ trong nhóm là OR lập tức lẻ.

In [5]:
print(2 | 4, "chẵn")                         # 6
print(2 | 4 | 3, "lẻ (vì có số 3 lẻ)")       # 7
print((2 | 4 | 6 | 8) & 1)                   # 0: toàn số chẵn
print((2 | 4 | 6 | 9) & 1)                   # 1: có số lẻ 9

6 chẵn
7 lẻ (vì có số 3 lẻ)
0
1


### 3.5. Duyệt mọi cặp: `combinations`

`itertools.combinations(nums, 2)` sinh mọi cặp `(nums[i], nums[j])` với `i < j` (không lặp cặp, không ghép phần tử với chính nó). Có `n(n - 1) / 2` cặp.

In [6]:
print(list(combinations([1, 2, 3, 4], 2)))
print(len(list(combinations(range(10), 2))), 10 * 9 // 2)      # 45 45

[(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
45 45


### 3.6. Đếm bằng `sum(điều_kiện ...)` và thoát sớm

`sum(x % 2 == 0 for x in nums)` đếm số phần tử chẵn (vì `True` cộng như `1`). Trong vòng lặp có thể `return` sớm ngay khi đạt điều kiện để khỏi đọc phần còn lại.

In [7]:
nums = [1, 2, 3, 4, 5]
print(sum(x % 2 == 0 for x in nums))          # 2
print([x for x in nums if x % 2 == 0])        # [2, 4]

2
[2, 4]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Thử vài nhóm với `nums = [1, 2, 3, 4, 5]` và nhìn bit thấp nhất của OR:

| nhóm chọn | OR (nhị phân) | bit thấp nhất | thỏa? |
|---|---|---|---|
| `1, 2` | `001 \| 010 = 011` | 1 | không |
| `2, 3` | `010 \| 011 = 011` | 1 | không |
| `1, 3` | `001 \| 011 = 011` | 1 | không |
| `2, 4` | `010 \| 100 = 110` | **0** | **có** |
| `2, 4, 5` | `010 \| 100 \| 101 = 111` | 1 | không |

Quan sát: **cứ có một số lẻ trong nhóm thì OR lẻ**. Nhóm thỏa chỉ gồm toàn số **chẵn**, và vì phải chọn **ít nhất hai** phần tử nên cần **ít nhất hai số chẵn**. Mà nếu đã có hai số chẵn thì **chọn đúng hai số đó** là đủ. Vậy đáp án là: **số lượng số chẵn có `>= 2` hay không**.

In [8]:
nums = [1, 2, 3, 4, 5]
for k in (2, 3):
    for group in combinations(nums, k):
        o = reduce(or_, group)
        mark = "<-- thỏa" if o % 2 == 0 else ""
        print(group, "OR =", o, bin(o), mark)

(1, 2) OR = 3 0b11 
(1, 3) OR = 3 0b11 
(1, 4) OR = 5 0b101 
(1, 5) OR = 5 0b101 
(2, 3) OR = 3 0b11 
(2, 4) OR = 6 0b110 <-- thỏa
(2, 5) OR = 7 0b111 
(3, 4) OR = 7 0b111 
(3, 5) OR = 7 0b111 
(4, 5) OR = 5 0b101 
(1, 2, 3) OR = 3 0b11 
(1, 2, 4) OR = 7 0b111 
(1, 2, 5) OR = 7 0b111 
(1, 3, 4) OR = 7 0b111 
(1, 3, 5) OR = 7 0b111 
(1, 4, 5) OR = 5 0b101 
(2, 3, 4) OR = 7 0b111 
(2, 3, 5) OR = 7 0b111 
(2, 4, 5) OR = 7 0b111 
(3, 4, 5) OR = 7 0b111 


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp

**Ý tưởng.** Thử mọi cặp `(i, j)` với `i < j`; nếu `(a | b)` chẵn (bit thấp nhất `0`) thì trả `True`. Vì một cặp cũng là một cách chọn "hai hoặc nhiều phần tử", việc chỉ xét cặp là đủ (ta sẽ chứng minh ở mục 10).

In [9]:
def has_trailing_zeros_pairs(nums: List[int]) -> bool:
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            if (nums[i] | nums[j]) & 1 == 0:               # bit thấp nhất của OR bằng 0
                return True
    return False


for nums, expected in examples:
    print(nums, "->", has_trailing_zeros_pairs(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 5] -> True (kỳ vọng True )
[2, 4, 8, 16] -> True (kỳ vọng True )
[1, 3, 5, 7, 9] -> False (kỳ vọng False )
[1, 2] -> False (kỳ vọng False )


In [10]:
# Xem các cặp được thử với ví dụ 1 (dừng khi tìm được)
nums = [1, 2, 3, 4, 5]
found = False
for i in range(len(nums)):
    for j in range(i + 1, len(nums)):
        o = nums[i] | nums[j]
        print(f"cặp ({nums[i]}, {nums[j]}): OR={o} ({bin(o)}) bit thấp nhất={o & 1}")
        if o & 1 == 0:
            found = True
            break
    if found:
        break
print("tìm thấy:", found)

cặp (1, 2): OR=3 (0b11) bit thấp nhất=1
cặp (1, 3): OR=3 (0b11) bit thấp nhất=1
cặp (1, 4): OR=5 (0b101) bit thấp nhất=1
cặp (1, 5): OR=5 (0b101) bit thấp nhất=1
cặp (2, 3): OR=3 (0b11) bit thấp nhất=1
cặp (2, 4): OR=6 (0b110) bit thấp nhất=0
tìm thấy: True


**Độ phức tạp (Cách 1).** `n(n - 1) / 2` cặp: **Time `O(n²)`**, **Space `O(1)`**.

**Nút thắt.** Mỗi cặp chỉ cần biết hai số **có chẵn không**; ta không cần so sánh cặp nào với cặp nào.

## 6. Ý tưởng thuật toán / bước đột phá

Từ mục 3.4 và 4:

1. OR của một nhóm là **chẵn ⇔ mọi phần tử trong nhóm chẵn**.
2. Vì phải chọn **từ hai phần tử trở lên**, tồn tại cách chọn ⇔ có **ít nhất hai số chẵn** (khi đó chọn đúng hai số chẵn đó).

Vậy chỉ cần **đếm số phần tử chẵn**: `count_even >= 2`. Một lần duyệt, `O(n)` thời gian, `O(1)` bộ nhớ, và thậm chí **dừng sớm** khi gặp số chẵn thứ hai.

## 7. Các cách tiếp theo

### Cách 2 — Đếm số phần tử chẵn

**Giải thích.** `count = sum(x % 2 == 0 for x in nums)` (hoặc `sum(1 for x in nums if x & 1 == 0)`), rồi trả `count >= 2`.

In [11]:
def has_trailing_zeros_count(nums: List[int]) -> bool:
    count = sum(1 for x in nums if x % 2 == 0)      # số phần tử chẵn
    return count >= 2


for nums, expected in examples:
    print(nums, "->", has_trailing_zeros_count(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 5] -> True (kỳ vọng True )
[2, 4, 8, 16] -> True (kỳ vọng True )
[1, 3, 5, 7, 9] -> False (kỳ vọng False )
[1, 2] -> False (kỳ vọng False )


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — Một lượt, thoát sớm với biến cờ

**Giải thích.** Dùng một cờ `seen_even = False`. Duyệt từng `x`: nếu `x` chẵn thì (nếu `seen_even` đã `True` ta có **hai** số chẵn nên `return True`) rồi đặt `seen_even = True`. Hết vòng lặp mà chưa trả thì `False`. Không cần đếm đến cuối.

In [12]:
def has_trailing_zeros_flag(nums: List[int]) -> bool:
    seen_even = False
    for x in nums:
        if x & 1 == 0:                               # x chẵn (bit thấp nhất = 0)
            if seen_even:
                return True                          # đã có một số chẵn trước đó -> đủ hai
            seen_even = True
    return False


for nums, expected in examples:
    print(nums, "->", has_trailing_zeros_flag(nums), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 5] -> True (kỳ vọng True )
[2, 4, 8, 16] -> True (kỳ vọng True )
[1, 3, 5, 7, 9] -> False (kỳ vọng False )
[1, 2] -> False (kỳ vọng False )


**Độ phức tạp Cách 3:** **Time `O(n)`** (tốt nhất `O(1)` nếu hai số chẵn nằm đầu mảng), **Space `O(1)`**.

### Cách 4 — Tổng quát hóa: OR có `>= t` bit `0` ở cuối ⇔ mọi số chia hết cho `2^t`

**Giải thích.** Bài hỏi `t = 1`. Tổng quát: OR của một nhóm có **ít nhất `t` bit `0` ở cuối** ⇔ **mọi số** trong nhóm có ít nhất `t` bit `0` ở cuối (OR không thể "tắt" bit `1`), tức mọi số **chia hết cho `2^t`**. Số bit `0` ở cuối của OR là **nhỏ nhất** trong các số bit `0` ở cuối của từng số. Vậy: tồn tại cách chọn ⇔ có **ít nhất hai** số chia hết cho `2^t`, tức `sum(x % (1 << t) == 0) >= 2`. Với `t = 1` trở về đúng bài này.

In [13]:
def has_trailing_zeros_general(nums: List[int], t: int = 1) -> bool:
    step = 1 << t                                     # 2^t
    return sum(1 for x in nums if x % step == 0) >= 2


for nums, expected in examples:
    print(nums, "->", has_trailing_zeros_general(nums, 1), "(kỳ vọng", expected, ")")

# Quan sát: số bit 0 ở cuối của OR = min các số bit 0 ở cuối
def trailing_zeros(x):
    return (x & -x).bit_length() - 1

a, b = 8, 12
print(f"{a}={bin(a)} có {trailing_zeros(a)} bit 0 cuối; {b}={bin(b)} có {trailing_zeros(b)}; OR={a | b}={bin(a | b)} có {trailing_zeros(a | b)}")
print("hai số chia hết cho 4 (t = 2) trong [3, 4, 8, 7]?", has_trailing_zeros_general([3, 4, 8, 7], 2))

[1, 2, 3, 4, 5] -> True (kỳ vọng True )
[2, 4, 8, 16] -> True (kỳ vọng True )
[1, 3, 5, 7, 9] -> False (kỳ vọng False )
[1, 2] -> False (kỳ vọng False )
8=0b1000 có 3 bit 0 cuối; 12=0b1100 có 2; OR=12=0b1100 có 2
hai số chia hết cho 4 (t = 2) trong [3, 4, 8, 7]? True


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`**.

In [14]:
# Code nộp LeetCode cho Cách 3 (một lượt, thoát sớm)
from typing import List

class Solution:
    def hasTrailingZeros(self, nums: List[int]) -> bool:
        seen_even = False
        for x in nums:
            if x % 2 == 0:
                if seen_even:
                    return True
                seen_even = True
        return False

## 8. TRACE từng bước

Ta theo dõi (a) Cách 3 với `[1, 3, 4, 7, 6, 9]` và (b) Cách 2 đếm số chẵn với `[1, 3, 5, 7, 9]`.

In [15]:
def trace_flag(nums):
    seen_even = False
    print("nums =", nums)
    for i, x in enumerate(nums):
        even = x & 1 == 0
        print(f"i={i} x={x:>2} {bin(x):>7} {'chẵn' if even else 'lẻ ':4} | seen_even trước = {seen_even!s:5}", end=" ")
        if even:
            if seen_even:
                print("-> số chẵn thứ hai: return True")
                return True
            seen_even = True
            print("-> số chẵn đầu tiên: seen_even = True")
        else:
            print("-> bỏ qua")
    print("=> hết mảng: False")
    return False


assert trace_flag([1, 3, 4, 7, 6, 9]) is True
assert trace_flag([1, 3, 5, 7, 9]) is False

nums = [1, 3, 4, 7, 6, 9]
i=0 x= 1     0b1 lẻ   | seen_even trước = False -> bỏ qua
i=1 x= 3    0b11 lẻ   | seen_even trước = False -> bỏ qua
i=2 x= 4   0b100 chẵn | seen_even trước = False -> số chẵn đầu tiên: seen_even = True
i=3 x= 7   0b111 lẻ   | seen_even trước = True  -> bỏ qua
i=4 x= 6   0b110 chẵn | seen_even trước = True  -> số chẵn thứ hai: return True
nums = [1, 3, 5, 7, 9]
i=0 x= 1     0b1 lẻ   | seen_even trước = False -> bỏ qua
i=1 x= 3    0b11 lẻ   | seen_even trước = False -> bỏ qua
i=2 x= 5   0b101 lẻ   | seen_even trước = False -> bỏ qua
i=3 x= 7   0b111 lẻ   | seen_even trước = False -> bỏ qua
i=4 x= 9  0b1001 lẻ   | seen_even trước = False -> bỏ qua
=> hết mảng: False


In [16]:
def trace_count(nums):
    count = 0
    for x in nums:
        if x % 2 == 0:
            count += 1
        print(f"x={x}: {'chẵn' if x % 2 == 0 else 'lẻ'} -> count={count}")
    ok = count >= 2
    print(f"count={count} >= 2? {ok}")
    return ok


assert trace_count([1, 3, 5, 7, 9]) is False
assert trace_count([2, 5, 8]) is True

x=1: lẻ -> count=0
x=3: lẻ -> count=0
x=5: lẻ -> count=0
x=7: lẻ -> count=0
x=9: lẻ -> count=0
count=0 >= 2? False
x=2: chẵn -> count=1
x=5: lẻ -> count=1
x=8: chẵn -> count=2
count=2 >= 2? True


Bảng tóm tắt (Cách 3, `[1, 3, 4, 7, 6, 9]`):

| `i` | `x` | chẵn? | `seen_even` trước | hành động |
|---|---|---|---|---|
| 0 | 1 | không | `False` | bỏ qua |
| 1 | 3 | không | `False` | bỏ qua |
| 2 | 4 | có | `False` | `seen_even = True` |
| 3 | 7 | không | `True` | bỏ qua |
| 4 | 6 | có | `True` | **`return True`** |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (một lượt, thoát sớm). Đúng class `Solution`, đúng method `hasTrailingZeros(self, nums: List[int]) -> bool`.

In [17]:
from typing import List

class Solution:
    def hasTrailingZeros(self, nums: List[int]) -> bool:
        seen_even = False                          # đã thấy một số chẵn chưa
        for x in nums:
            if x % 2 == 0:                         # x chẵn: bit thấp nhất bằng 0
                if seen_even:
                    return True                    # số chẵn thứ hai -> chọn hai số này
                seen_even = True
        return False                               # có không quá một số chẵn


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.hasTrailingZeros(nums))

[1, 2, 3, 4, 5] -> True
[2, 4, 8, 16] -> True
[1, 3, 5, 7, 9] -> False
[1, 2] -> False


## 10. Vì sao thuật toán đúng

- **OR giữ mọi bit `1`.** Bit thấp nhất của `a | b | ...` bằng `1` nếu **bất kỳ** số nào trong nhóm có bit thấp nhất `1` (lẻ), và bằng `0` chỉ khi **mọi** số trong nhóm chẵn. Vậy OR của nhóm có bit `0` ở cuối ⇔ nhóm **toàn số chẵn**.
- **Điều kiện cần.** Nếu tồn tại cách chọn thỏa (nhóm có `>= 2` phần tử, toàn số chẵn) thì `nums` có ít nhất `2` số chẵn.
- **Điều kiện đủ.** Nếu `nums` có ít nhất `2` số chẵn, chọn đúng hai số chẵn đó (hai phần tử, OR chẵn): thỏa đề.
- **Chỉ xét cặp là đủ.** Mọi nhóm thỏa (toàn chẵn, `>= 2` phần tử) chứa một cặp thỏa. Do đó vét cạn mọi cặp (Cách 1) cho cùng đáp án với vét cạn mọi tập con (ta sẽ kiểm chứng bằng code).
- **Thoát sớm hợp lệ.** Cờ `seen_even` cho biết đã thấy đúng một số chẵn; gặp số chẵn tiếp theo là đã có hai, thỏa điều kiện nên `return True` ngay mà không cần xem phần còn lại.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- **Cách 1 (mọi cặp):** Time `O(n²)`, Space `O(1)`.
- **Cách 2 (đếm số chẵn):** Time `O(n)`, Space `O(1)`.
- **Cách 3 (cờ, thoát sớm):** Time `O(n)` (tốt nhất `O(1)`), Space `O(1)`.
- **Cách 4 (tổng quát `2^t`):** Time `O(n)`, Space `O(1)`.
- Mọi cách tối ưu phải xem ít nhất một phần mảng để tìm hai số chẵn; `O(n)` là tối ưu trong tệ nhất. Ở đây `n <= 100`.

## 12. Bộ test

In [18]:
sol = Solution()
impls = {
    "pairs": has_trailing_zeros_pairs,
    "count": has_trailing_zeros_count,
    "flag": has_trailing_zeros_flag,
    "general": has_trailing_zeros_general,
    "submit": sol.hasTrailingZeros,
}

for nums, expected in examples:
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got is expected, (key, nums, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [19]:
def check_all(nums, expected, label):
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got is expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([2, 4], True, "n = 2, hai số chẵn")
check_all([2, 3], False, "n = 2, một chẵn một lẻ")
check_all([1, 3], False, "n = 2, hai số lẻ")
check_all([100, 100], True, "hai số chẵn bằng nhau (chọn hai vị trí khác nhau)")
check_all([2, 1, 1, 1, 1], False, "chỉ có một số chẵn")
check_all([1, 1, 1, 1, 2, 4], True, "hai số chẵn ở cuối")
check_all([2, 1, 1, 1, 4], True, "hai số chẵn cách xa nhau")
check_all([1] * 100, False, "n = 100 toàn số lẻ")
check_all([2] * 100, True, "n = 100 toàn số chẵn")
check_all([1] * 99 + [100], False, "n = 100, chỉ số chẵn duy nhất ở cuối")
check_all([99] * 98 + [98, 100], True, "n = 100, hai số chẵn ở cuối")

PASS n = 2, hai số chẵn -> True
PASS n = 2, một chẵn một lẻ -> False
PASS n = 2, hai số lẻ -> False
PASS hai số chẵn bằng nhau (chọn hai vị trí khác nhau) -> True
PASS chỉ có một số chẵn -> False
PASS hai số chẵn ở cuối -> True
PASS hai số chẵn cách xa nhau -> True
PASS n = 100 toàn số lẻ -> False
PASS n = 100 toàn số chẵn -> True
PASS n = 100, chỉ số chẵn duy nhất ở cuối -> False
PASS n = 100, hai số chẵn ở cuối -> True


### 12.2. Tính chất và phản ví dụ

In [20]:
# Chuẩn vàng: thử MỌI tập con có từ 2 phần tử trở lên (n nhỏ)
def brute_subsets(nums):
    n = len(nums)
    for mask in range(1, 1 << n):
        if bin(mask).count("1") < 2:
            continue
        o = 0
        for i in range(n):
            if mask >> i & 1:
                o |= nums[i]
        if o & 1 == 0:
            return True
    return False


for nums, expected in examples:
    assert brute_subsets(nums) is expected
print("PASS vét cạn mọi tập con khớp ví dụ")

random.seed(11)
for _ in range(1500):
    n = random.randint(2, 10)
    nums = [random.randint(1, 100) for _ in range(n)]
    assert brute_subsets(nums) == sol.hasTrailingZeros(nums), nums
print("PASS vét cạn mọi tập con == đáp án (chỉ cần cặp là đủ)")

# Tính chất 1: thêm số lẻ vào mảng không đổi đáp án
for _ in range(300):
    nums = [random.randint(1, 100) for _ in range(random.randint(2, 50))]
    odd = random.choice([x for x in range(1, 101) if x % 2 == 1])
    assert sol.hasTrailingZeros(nums) == sol.hasTrailingZeros(nums + [odd])
print("PASS thêm số lẻ không đổi đáp án")

# Tính chất 2: thứ tự không quan trọng
for _ in range(300):
    nums = [random.randint(1, 100) for _ in range(random.randint(2, 50))]
    shuffled = nums[:]
    random.shuffle(shuffled)
    assert sol.hasTrailingZeros(nums) == sol.hasTrailingZeros(shuffled)
print("PASS thứ tự không quan trọng")

# Tính chất 3: số bit 0 ở cuối của OR = min của các số bit 0 ở cuối (Cách 4)
def tz(x):
    return (x & -x).bit_length() - 1

for _ in range(500):
    group = [random.randint(1, 100) for _ in range(random.randint(2, 6))]
    assert tz(reduce(or_, group)) == min(tz(x) for x in group)
print("PASS tz(OR) = min tz")

# Tính chất 4: đầu vào không bị sửa
nums = [1, 2, 3, 4, 5]
copy = list(nums)
for key, fn in impls.items():
    fn(nums)
assert nums == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: chỉ cần một số chẵn
def sai_mot_chan(nums):
    return any(x % 2 == 0 for x in nums)

assert sai_mot_chan([2, 1, 1]) is True and sol.hasTrailingZeros([2, 1, 1]) is False
print("PASS phản ví dụ: [2, 1, 1] chỉ có một số chẵn nên đáp án False")

# Phản ví dụ 2: OR của tất cả phần tử
def sai_or_tat_ca(nums):
    return reduce(or_, nums) % 2 == 0

assert sai_or_tat_ca([2, 4, 1]) is False and sol.hasTrailingZeros([2, 4, 1]) is True
print("PASS phản ví dụ: OR cả mảng [2, 4, 1] là lẻ nhưng chọn {2, 4} thì được")

PASS vét cạn mọi tập con khớp ví dụ
PASS vét cạn mọi tập con == đáp án (chỉ cần cặp là đủ)
PASS thêm số lẻ không đổi đáp án
PASS thứ tự không quan trọng
PASS tz(OR) = min tz
PASS đầu vào không bị sửa
PASS phản ví dụ: [2, 1, 1] chỉ có một số chẵn nên đáp án False
PASS phản ví dụ: OR cả mảng [2, 4, 1] là lẻ nhưng chọn {2, 4} thì được


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [21]:
random.seed(2980)
for trial in range(3000):
    n = random.randint(2, 100)
    mode = random.choice(["mixed", "mostly_odd", "mostly_even"])
    if mode == "mixed":
        nums = [random.randint(1, 100) for _ in range(n)]
    elif mode == "mostly_odd":
        nums = [random.choice(range(1, 100, 2)) for _ in range(n)]
        if random.random() < 0.5:
            nums[random.randrange(n)] = random.choice(range(2, 101, 2))      # đúng 0 hoặc 1 số chẵn
    else:
        nums = [random.choice(range(2, 101, 2)) for _ in range(n)]
    ref = has_trailing_zeros_pairs(list(nums))
    for key, fn in impls.items():
        assert fn(list(nums)) is ref, (key, nums)
print("PASS 3000 bộ ngẫu nhiên - mọi cách khớp nhau")

PASS 3000 bộ ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tưởng chỉ cần có một số chẵn

In [22]:
def sai_mot_chan_2(nums):
    return any(x % 2 == 0 for x in nums)


print("sai  :", sai_mot_chan_2([1, 2, 3]), "| đúng:", Solution().hasTrailingZeros([1, 2, 3]))

sai  : True | đúng: False


**Vì sao sai:** đề bắt buộc chọn **từ hai phần tử trở lên**. Với `[1, 2, 3]` chỉ có một số chẵn (`2`); mọi nhóm hai phần tử đều chứa một số lẻ nên OR lẻ. Phải có **ít nhất hai** số chẵn.

### Lỗi 2: OR tất cả các phần tử

In [23]:
def sai_or_het(nums):
    o = 0
    for x in nums:
        o |= x
    return o % 2 == 0


print("sai  :", sai_or_het([2, 4, 1]), "| đúng:", Solution().hasTrailingZeros([2, 4, 1]))

sai  : False | đúng: True


**Vì sao sai:** đề cho phép **chọn nhóm con**, không bắt buộc dùng tất cả. Một số lẻ lọt vào làm OR cả mảng lẻ, nhưng ta chỉ việc **bỏ nó ra** và chọn các số chẵn. Đừng OR toàn bộ khi đề nói "chọn".

### Lỗi 3: Dùng `> 2` thay vì `>= 2`

In [24]:
def sai_lon_hon_2(nums):
    return sum(1 for x in nums if x % 2 == 0) > 2


print("sai  :", sai_lon_hon_2([2, 4]), "| đúng:", Solution().hasTrailingZeros([2, 4]))

sai  : False | đúng: True


**Vì sao sai:** chỉ cần **đúng hai** số chẵn đã chọn được (một cặp). `> 2` đòi tới ba số chẵn nên bỏ sót trường hợp hai số. Điều kiện đúng là `>= 2`.

### Lỗi 4: Nhầm "chữ số 0 ở cuối" thập phân với nhị phân

In [25]:
def sai_thap_phan(nums):
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            if (nums[i] | nums[j]) % 10 == 0:          # 0 ở cuối trong hệ thập phân
                return True
    return False


print("sai  :", sai_thap_phan([2, 4]), "| đúng:", Solution().hasTrailingZeros([2, 4]))

sai  : False | đúng: True


**Vì sao sai:** đề nói về **biểu diễn nhị phân**. `2 | 4 = 6` có bit `0` ở cuối (`110`) nên đáp án `True`, trong khi `6 % 10 != 0` (thập phân không tận cùng bằng `0`). Kiểm tra đúng là `% 2 == 0` hoặc `& 1 == 0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "OR có bit `0` ở cuối" → nghĩ ngay **tính chẵn lẻ**: OR chẵn ⇔ mọi số chẵn.
- Công thức nhớ: **chọn nhóm con để OR chẵn ⇔ nhóm toàn số chẵn**; có `>= 2` số chẵn là chọn được.
- Với AND: bit của AND bằng `1` chỉ khi **mọi** số có bit `1`. Với OR: bit bằng `0` chỉ khi **mọi** số có bit `0`. Nhớ hai quy tắc này để suy luận nhanh.
- Số bit `0` ở cuối: `(x & -x).bit_length() - 1`; OR lấy **min** các số này; AND thì **không nhỏ hơn max** của chúng.
- Khi bài nói "chọn nhóm con tùy ý", cân nhắc đáp án tổng quát chỉ cần **cặp** thay vì thử mọi tập con (`2ⁿ`).
- Kiểm chứng giả thuyết bằng vét cạn mọi tập con với `n` nhỏ trước khi nộp.

## 15. Biến thể của bài

1. Cần OR có **ít nhất `t`** bit `0` ở cuối (Cách 4).
2. Chọn **ba** phần tử trở lên (hoặc đúng `k` phần tử).
3. Chọn tập con **không rỗng** (một phần tử cũng được): đáp án là `có số chẵn`.
4. Thay OR bằng **AND**: AND có bit `0` ở cuối khi **ít nhất một** số chẵn.
5. Thay OR bằng **XOR**: XOR chẵn khi số phần tử lẻ trong nhóm là chẵn.
6. Đếm số **cặp** `(i < j)` mà OR có bit `0` ở cuối.

Thử biến thể 1, 3 và 6:

In [26]:
def at_least_t_trailing_zeros(nums, t):
    return sum(1 for x in nums if x % (1 << t) == 0) >= 2


def has_with_single_allowed(nums):
    return any(x % 2 == 0 for x in nums)


def count_good_pairs(nums):
    evens = sum(1 for x in nums if x % 2 == 0)
    return evens * (evens - 1) // 2


def count_good_pairs_brute(nums):
    return sum(1 for a, b in combinations(nums, 2) if (a | b) & 1 == 0)


print("1. t = 2 trên [4, 8, 3, 6]:", at_least_t_trailing_zeros([4, 8, 3, 6], 2))      # True: 4 và 8
print("3. cho phép chọn một phần tử, [1, 3, 2]:", has_with_single_allowed([1, 3, 2]))  # True
print("6. số cặp tốt của [1, 2, 3, 4, 6]:", count_good_pairs([1, 2, 3, 4, 6]))         # 3
assert at_least_t_trailing_zeros([4, 8, 3, 6], 2) is True
assert at_least_t_trailing_zeros([4, 6, 3, 2], 2) is False
assert has_with_single_allowed([1, 3, 2]) is True
for _ in range(200):
    arr = [random.randint(1, 100) for _ in range(random.randint(2, 30))]
    assert count_good_pairs(arr) == count_good_pairs_brute(arr)
print("PASS đếm cặp bằng công thức C(số chẵn, 2) khớp vét cạn")

1. t = 2 trên [4, 8, 3, 6]: True
3. cho phép chọn một phần tử, [1, 3, 2]: True
6. số cặp tốt của [1, 2, 3, 4, 6]: 3
PASS đếm cặp bằng công thức C(số chẵn, 2) khớp vét cạn


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cặp | `(a \| b) & 1 == 0` | `O(n²)` | `O(1)` | hiểu đề, đáp án chuẩn |
| 2. Đếm số chẵn | `count >= 2` | `O(n)` | `O(1)` | ngắn, dễ hiểu |
| 3. Cờ `seen_even`, thoát sớm | gặp số chẵn thứ hai thì dừng | `O(n)` | `O(1)` | **cách nộp bài**, nhanh nhất trong trường hợp tốt |
| 4. Tổng quát `x % 2^t == 0` | OR có `>= t` bit 0 cuối | `O(n)` | `O(1)` | biến thể nhiều bit 0 ở cuối |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `5 & 1`, `bin(6 | 8)`, `3 | 4 | 5`, `(12 & -12)`, `Solution().hasTrailingZeros([6, 1, 8])`.

**Bài 2 (tính tay).** Với `nums = [3, 5, 6, 7, 10]`: liệt kê các số chẵn; chọn ra một cặp thỏa và viết OR dưới dạng nhị phân; sau đó với `nums = [3, 5, 6, 7, 9]` giải thích vì sao `False`.

**Bài 3 (điền TODO).** Hoàn thành `has_trailing_zeros_todo(nums)` theo Cách 2.

In [27]:
def has_trailing_zeros_todo(nums):
    # TODO 1: count = số phần tử chẵn trong nums
    # TODO 2: trả về True nếu count >= 2, ngược lại False
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `has_trailing_zeros_flag2(nums)` theo Cách 3 (cờ `seen_even`, thoát sớm) mà không nhìn code mẫu. Kiểm tra với bốn ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `has_k_numbers(nums, k)`: có thể chọn **`k` phần tử trở lên** sao cho OR chẵn không (`k >= 1`)? (b) Viết `min_pair_trailing_zeros(nums)` trả về **số bit `0` ở cuối lớn nhất** có thể đạt được khi OR của một cặp (trả `0` nếu không có cặp nào thỏa). (c) Viết `has_and_trailing_zeros(nums)`: tồn tại cặp có **AND** chẵn không?

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `1`, `0b1110`, `7`, `4`, `True`. Bài 3: `count = sum(1 for x in nums if x % 2 == 0)` rồi `return count >= 2`.
- **Mức 2 (vừa):** Bài 2: `[3, 5, 6, 7, 10]` có hai số chẵn `6` và `10`: `6 | 10 = 110 | 1010 = 1110 = 14` (kết thúc bằng bit `0`). `[3, 5, 6, 7, 9]` chỉ có một số chẵn (`6`), mọi cặp gồm số lẻ nên OR lẻ. Bài 4: xem Cách 3.
- **Mức 3 (sâu):** Bài 5(a): nhóm `k` phần tử trở lên có OR chẵn ⇔ nhóm toàn số chẵn, nên chỉ cần có **ít nhất `k` số chẵn** (`evens >= k`); `k = 2` trở về đúng đề. 5(b): lấy mọi số chẵn, tìm hai số có `min(tz(a), tz(b))` lớn nhất: sắp xếp `tz` giảm dần rồi lấy số thứ hai. 5(c): AND có bit thấp nhất `0` khi **ít nhất một** trong hai số chẵn: tồn tại cặp ⇔ có số chẵn và `n >= 2` (hoặc có hai số khi `n >= 2`; nếu mọi số đều lẻ thì AND lẻ).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert (5 & 1) == 1 and bin(6 | 8) == "0b1110" and (3 | 4 | 5) == 7 and (12 & -12) == 4
assert Solution().hasTrailingZeros([6, 1, 8]) is True

# Bài 2
assert [x for x in [3, 5, 6, 7, 10] if x % 2 == 0] == [6, 10]
assert bin(6 | 10) == "0b1110"
assert Solution().hasTrailingZeros([3, 5, 6, 7, 10]) is True
assert Solution().hasTrailingZeros([3, 5, 6, 7, 9]) is False

# Bài 3
def has_trailing_zeros_todo_ans(nums):
    count = sum(1 for x in nums if x % 2 == 0)
    return count >= 2

for nums, expected in examples:
    assert has_trailing_zeros_todo_ans(nums) is expected

# Bài 4
def has_trailing_zeros_flag2(nums):
    seen = False
    for x in nums:
        if x % 2 == 0:
            if seen:
                return True
            seen = True
    return False

for nums, expected in examples:
    assert has_trailing_zeros_flag2(nums) is expected

# Bài 5(a)
def has_k_numbers(nums, k):
    evens = sum(1 for x in nums if x % 2 == 0)
    return evens >= k                                   # chọn k số chẵn là đủ (nhóm k phần tử trở lên)

assert has_k_numbers([1, 2, 3], 1) is True
assert has_k_numbers([1, 2, 3], 2) is False
assert has_k_numbers([2, 4, 6, 1], 3) is True
assert has_k_numbers([2, 4, 1], 3) is False

# Bài 5(b)
def min_pair_trailing_zeros(nums):
    def tz(x):
        return (x & -x).bit_length() - 1
    t = sorted((tz(x) for x in nums if x % 2 == 0), reverse=True)
    return t[1] if len(t) >= 2 else 0

assert min_pair_trailing_zeros([4, 8, 3, 6]) == 2            # 4 và 8 cùng >= 2 bit 0
assert min_pair_trailing_zeros([2, 3, 5]) == 0
assert min_pair_trailing_zeros([16, 2, 24]) == 3             # 16 (4) và 24 (3): min = 3
for _ in range(200):
    arr = [random.randint(1, 100) for _ in range(random.randint(2, 15))]
    ok_pairs = [((a | b) & -(a | b)).bit_length() - 1 for a, b in combinations(arr, 2) if (a | b) % 2 == 0]
    assert min_pair_trailing_zeros(arr) == (max(ok_pairs) if ok_pairs else 0)

# Bài 5(c)
def has_and_trailing_zeros(nums):
    return len(nums) >= 2 and any(x % 2 == 0 for x in nums)

assert has_and_trailing_zeros([1, 3, 2]) is True
assert has_and_trailing_zeros([1, 3, 5]) is False
for _ in range(200):
    arr = [random.randint(1, 100) for _ in range(random.randint(2, 15))]
    assert has_and_trailing_zeros(arr) == any((a & b) % 2 == 0 for a, b in combinations(arr, 2))
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (OR chẵn ⇔ mọi số chẵn; cần ít nhất hai số chẵn)

```python
from typing import List

class Solution:
    def hasTrailingZeros(self, nums: List[int]) -> bool:
        evens = 0
        for x in nums:
            if x % 2 == 0:
                evens += 1
        return evens >= 2
```

**Checklist**

- [ ] Hiểu "bit `0` ở cuối" là về **nhị phân** (số **chẵn**), không phải thập phân?
- [ ] Đòi **ít nhất hai** số chẵn (`>= 2`), không phải một, không phải `> 2`?
- [ ] Không OR **cả mảng** (đề cho phép chọn nhóm con)?
- [ ] Nếu thoát sớm: chỉ `return True` ở số chẵn **thứ hai**, và `return False` sau vòng lặp?
- [ ] Đã thử: hai số lẻ, một chẵn một lẻ, hai chẵn, `n = 100`, chỉ một số chẵn duy nhất?
- [ ] Skeleton đúng `class Solution` / `def hasTrailingZeros(self, nums: List[int]) -> bool`?